In [133]:
import json

path = "../../../synthetic_sampling/samples/synthetic_samples_complete.json"
with open(path, "r", encoding="utf-8") as f:
    total_data = json.load(f)

with open("../../../synthetic_sampling/samples/for_synth_sampling.json", "r", encoding="utf-8") as f:
    real_smpls = json.load(f)

# paper_name -> {"document": full_text, "spans": [...]}
by_paper = {}

for paper_name, samples in real_smpls.items():
    for s in samples:
        if s["label"] != "Unsupported claim":
            continue
        
        if paper_name not in by_paper:
            by_paper[paper_name] = {
                "document": samples[0]["full_text"] if samples else "",
                "spans": [],
            }

        for s in samples:
            by_paper[paper_name]["spans"].append(
                {
                    "span": s["text"],
                    "start": s["start"],
                    "end": s["end"],
                    "label": s["label"],
                }
            )

# final list (one item per paper)
total_real = list(by_paper.values())

In [128]:
synth_data = total_data['Unsupported claim']

In [137]:
import spacy
from tqdm import tqdm
import json

nlp = spacy.load("en_core_web_sm")

def split_sentences_with_offsets(text):
    doc = nlp(text)
    out = []
    for sent in doc.sents:
        sent_text = text[sent.start_char:sent.end_char]
        if sent_text.strip() == "":
            continue
        out.append({"text": sent_text, "start": sent.start_char, "end": sent.end_char})
    return out

def collecting_spans(spans, sents):
    data = []

    for span in spans:
        span_start_doc = span["start"]
        span_end_doc = span["end"]
        span_text = span.get("span") or span.get("text")  # optional, but helps a lot

        for sent in sents:
            sent_start_doc = sent["start"]
            sent_end_doc = sent["end"]

            if span_end_doc <= sent_start_doc or span_start_doc >= sent_end_doc:
                continue

            # doc -> sentence offsets
            local_start = max(0, span_start_doc - sent_start_doc)
            local_end = min(len(sent["text"]), span_end_doc - sent_start_doc)

            extracted = sent["text"][local_start:local_end]

            # If we have the true span string and it doesn't match, re-align by searching
            if span_text and extracted != span_text:
                idx = sent["text"].find(span_text)
                if idx != -1:
                    local_start = idx
                    local_end = idx + len(span_text)
                    extracted = sent["text"][local_start:local_end]

            data.append({
                "text": sent["text"],
                "span": extracted,
                "start": local_start,
                "end": local_end,
            })

    return data

In [138]:
processed_synth = []
for sample in tqdm(synth_data):
    sentences = split_sentences_with_offsets(sample['document'])
    spans = sample['spans']
    matched_span_sent = collecting_spans(spans, sentences)
    processed_synth.append(matched_span_sent)

processed_eval = []
for sample in tqdm(total_real):
    sentences = split_sentences_with_offsets(sample['document'])
    spans = sample['spans']
    matched_span_sent = collecting_spans(spans, sentences)
    processed_eval.append(matched_span_sent)

100%|██████████| 30/30 [00:02<00:00, 10.42it/s]


In [140]:
import random


def sentence_overlaps_any_span(sent_start, sent_end, spans):
    for sp in spans:
        sp_start = sp["start"]
        sp_end = sp["end"]
        if not (sp_end <= sent_start or sp_start >= sent_end):
            return True
    return False


def build_sentence_dataset(items, neg_ratio=1.0, seed=13):
    """
    items: [{"document": str, "spans": [{"start":int,"end":int,...}, ...]}, ...]
    neg_ratio: how many negatives per positive to keep (1.0 => ~1:1)
    """
    rng = random.Random(seed)
    out = []

    for sample in items:
        doc_text = sample["document"]
        spans = sample.get("spans", [])
        sents = split_sentences_with_offsets(doc_text)  # [{"text","start","end"}, ...]

        positives = []
        negatives = []

        for s in sents:
            sent_text = s["text"]
            sent_start = s["start"]
            sent_end = s["end"]

            y = 1 if sentence_overlaps_any_span(sent_start, sent_end, spans) else 0
            ex = {
                "text": sent_text,
                "label": y,
                # optional debug metadata:
                # "doc_start": sent_start,
                # "doc_end": sent_end,
            }

            if y == 1:
                positives.append(ex)
            else:
                negatives.append(ex)

        out.extend(positives)

        # sample negatives to balance
        target_neg = int(round(len(positives) * float(neg_ratio)))
        if target_neg <= 0:
            continue

        if target_neg >= len(negatives):
            out.extend(negatives)
        else:
            out.extend(rng.sample(negatives, target_neg))

    return out

In [142]:
eval_dataset = build_sentence_dataset(total_real, neg_ratio=1.0, seed=42)  
train_dataset = build_sentence_dataset(synth_data, neg_ratio=1.0, seed=42)  

In [47]:
format_def = """Issues in citation formatting such as a missing bracket and using the wrong style of citing.
    a) Due to preprocessing errors of the source dataset, some words contain hyphens that do not require it, and some are missing hyphens where it is required. Please ignore these types of formatting issues.
    b) Highlight the word/citation in which the formatting issue occurs in and not only the issue within the word/citation.
    c) Formatting issues appear as either citations or parts of a citation.
    Examples of formatting issues include:
        i) Narrative citation missing year: “Vatswani et al.” -> should be “Vatswani et al. (2020)”
        ii) Wrong citation style: “In (Vatswani et al., 2019)” -> should be “in Vatswani et al. (2019)”
        iii) Wrong use of footnotes: "Vastwani et al. 1" -> should include the year or be reformatted as a proper footnote."""

unsupp_def = """claim about prior work or statistics w/o citation or evidence. 
    a) The author should cite at every first mention of a study, paper, shared task, competition or dataset.
    b) Specific information to a niche topic, despite sounding like it should be known in that topic of study, should be cited.
    c) If a claim is made and is obvious to be a natural deduction from previous statements through common sense (i.e not requiring expert knowledge), then this claim does not fall under ‘Unsupported claim’. For example:
        i) “However, creating a large and suitable set of questions for supporting narrative comprehension is both time-consuming and cognitively demanding.” -> it is obvious that creating a dataset is time consuming and mentally demanding.
    d) Any mention of “recent works” should be backed up with citations to the works.
    e) Unsupported claim issues appear as segments, phrases, sub-sentences or full sentences.
    Examples of unsupported claims include:
        i) Missing citations for mentions of 'recent works': “and there are many recent works that explore this topic”,
        ii) Mention of a previous work and claim without citation: “..., while in a previous study, the authors claim …”,
        iii) Mentioning of a specific setup of a task without citation to the work: ".. BERT was used in an AES task trained on essays .." """

lacksynth_def = """occurs when either:
    a) The author describes or cites papers without connecting them to their own work/argument 
    b) Or only follows up the summary of previous works with their own contribution without explicitly highlighting the gap their work intends to research.
    c) It does not articulate the author's perspective or motivation.
    d) A lack of argument/opinion in the first paragraph is permissible as it serves to be the foundation of the author's argument 
    e) Lacks synthesis issues appear either as single sentences or multiple sentences.
    Examples of lack of synthesis include:
        i) No elaboration of own contribution/argument:"Following early neural approaches to question answering, many subsequent studies adopt a pipeline architecture consisting of retrieval and comprehension components. The retrieval component focuses on identifying relevant documents or passages from a large corpus, while the comprehension component extracts an answer span from the retrieved text. Initial models relied on recurrent neural networks with attention mechanisms to encode questions and contexts (Seo et al., 2017; Wang et al., 2017)."
        ii)  No explanation of the cited works and relation to their own work: “Recently, several studies have explored the use of prompting techniques with pre-trained language models to influence model outputs or access latent knowledge (Brown et al., 2020; Gao et al., 2021; Liu et al., 2021; Wei et al., 2022).” """

coherence_def = """connection between cited works is abrupt, lacking relation to each other. It is unclear how one mentioned work is relevant to a prior mentioned work. 
    a) Sentences are not transitioned from one to another.
    b) The relationship between sentences describing papers is implied but not explicitly stated.
    c) Coherence issues appear only as multiple sentences.
    Examples of coherence issues include:
        i) Relation between mentioned works is not explicit: “Smith (2020) identified a relationship between personal belief systems and ethical decision-making frameworks. Moral foundation theory proposes several core dimensions of moral reasoning, including harm, fairness, and authority (Jones, 2015). Audience adaptation has been explored in computational argumentation. Lee et al. (2019) applied moral categories to argument generation tasks. Human annotators often disagree when labeling moral dimensions in text (Nguyen et al., 2018).”
        ii) Lack of transitions between sentences: “Recent studies have explored various techniques for enhancing model performance. Smith et al. (2020) introduced a novel architecture that significantly improves accuracy on benchmark datasets. Additionally, Johnson and Lee (2019) proposed a data augmentation method that increases training data diversity.” 
        iii) No explanation of the cited works and relation to their own work: “Recently, several studies have explored the use of prompting techniques with pre-trained language models to influence model outputs or access latent knowledge (Brown et al., 2020; Gao et al., 2021; Liu et al., 2021; Wei et al., 2022).” """

In [ ]:
def process_data(data):
    processed_data = {}

    for category, samples in data.items():
        if category == 'Unsupported_claim':
            definition = unsupp_def
        elif category == 'Lacks_synthesis':
            definition = lacksynth_def
        elif category == 'Coherence':
            definition = coherence_def
        elif category == 'Format':
            definition = format_def
        else:
            continue

        processed_data.setdefault(category, [])

        for sample in samples:
            span_text = sample["span"].strip()

            prompt = f"""You are an expert annotator.
Task:
Return ONLY the spans from the document that match the definition below.
Each span MUST be wrapped exactly like this:

<span>your span here</span>

Do not output anything else.

Definition:
{definition}

Text:
{sample['document']}
"""

            completion = f"<span>{span_text}</span>"

            processed_data[category].append({
                "prompt": prompt,
                "completion": completion
            })

    return processed_data

train_data = process_data(train_data)
dev_data = process_data(dev_data)
eval_data = process_data(eval_data)